# 6.2 — Universal Approximation

Universal approximation says that a wide enough neural network with a nonlinear hidden layer can approximate any continuous function on a closed, bounded input range. This lesson builds that claim from scratch in NumPy: affine signals create movable breakpoints, nonlinear activations turn those signals into reusable basis pieces, output weights add the pieces together, and training tries to discover one such approximation even though the theorem only promises that one exists.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build universal approximation one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is shown so the theorem feels like arithmetic instead of magic. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vectorized activations, and linear algebra from scratch.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for deterministic toy fits.

### 1. What the theorem is really approximating

The theorem is about **continuous functions on a compact domain**. We will use a smooth target on the closed interval `[-1, 1]`, then ask a one-hidden-layer network to imitate its values on a grid. The compact interval matters: on a bounded range, a continuous curve cannot hide infinitely violent behavior, so a finite collection of local nonlinear pieces can cover it.

In [ ]:
x_w = np.linspace(-1.0, 1.0, 201)[:, None]  # compact domain as a column vector.
y_w = np.sin(3 * x_w) + 0.3 * x_w ** 2      # continuous target values on that domain.

print("x shape:", x_w.shape, "y shape:", y_w.shape)  # inspect the supervised pairs.
print("target range:", round(float(y_w.min()), 3), "to", round(float(y_w.max()), 3))

▶ What you'll see: 201 input-output pairs sampled from a bounded, smooth target curve.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(x_w[:, 0], y_w[:, 0], color="black", label="target f(x)")
plt.title("1: continuous target on a compact interval")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.legend(); plt.show()

▶ What you'll see: a smooth curve with no jumps; this is the kind of object the theorem covers.

*Why it's done this way:* the theorem is not a promise about arbitrary wild functions everywhere on the real line. It is a compact-domain statement: once the input lives in a finite interval and the target is continuous, local errors can be made uniformly small by enough nonlinear pieces.

### 2. A nonlinear hidden unit is an adjustable basis piece

A hidden unit first forms an affine signal `z = wx + b`, then applies a nonlinearity. With ReLU, `max(0, z)`, the unit is exactly zero on one side of its breakpoint and linear on the other. Moving `w` and `b` moves the breakpoint; scaling the output weight controls how much that piece contributes.

In [ ]:
x2_w = np.linspace(-1, 1, 9)[:, None]  # a tiny grid so the arithmetic is visible.
w2_w, b2_w = 4.0, -0.5                # affine parameters for one hidden unit.
z2_w = w2_w * x2_w + b2_w             # pre-activation signal.
h2_w = np.maximum(0.0, z2_w)          # ReLU basis piece.

print("z values:", np.round(z2_w[:, 0], 2))
print("ReLU(z):", np.round(h2_w[:, 0], 2))

▶ What you'll see: negative affine values are clipped to zero, while positive ones pass through.

In [ ]:
break2_w = -b2_w / w2_w  # solve wx+b=0 for the breakpoint.

print("breakpoint:", round(float(break2_w), 3))

assert round(float(break2_w), 3) == 0.125
plt.figure(figsize=(5, 3))
plt.plot(x2_w[:, 0], z2_w[:, 0], "--", label="affine wx+b")
plt.plot(x2_w[:, 0], h2_w[:, 0], marker="o", label="ReLU(wx+b)")
plt.axvline(break2_w, color="gray", linestyle=":", label="breakpoint")
plt.title("2: one nonlinear basis piece"); plt.xlabel("x"); plt.legend(); plt.show()

▶ What you'll see: the ReLU stays flat until the learned breakpoint, then becomes a ramp.

*Why it's done this way:* a purely linear model can only draw one global line, but a nonlinear unit creates a local change in slope. Universal approximation is built from many such slope changes; the output layer adds them into a flexible piecewise-linear curve.

### 3. Weighted sums of hidden units make bends

A one-hidden-layer network in one dimension has the form `sum_j a_j phi(w_j x + b_j) + c`. The hidden layer creates several nonlinear basis pieces, and the output layer adds them with positive or negative weights. With ReLU units, every hidden unit can introduce one new bend, so a width-`m` network can build a piecewise-linear approximation.

In [ ]:
x3_w = np.linspace(-1, 1, 201)[:, None]
centers3_w = np.array([-0.75, -0.25, 0.25, 0.75])  # where we want slope changes.
w3_w = np.ones((1, 4)) * 8.0                       # steepness shared by all hidden units.
b3_w = -8.0 * centers3_w.reshape(1, -1)             # makes each breakpoint land at its center.
H3_w = np.maximum(0.0, x3_w @ w3_w + b3_w)           # hidden activations: 201 by 4.

print("hidden shape:", H3_w.shape)
print("activation sample at x=0:", np.round(H3_w[100], 3))

▶ What you'll see: four hidden columns, each a ramp that begins at a different input location.

In [ ]:
a3_w = np.array([[0.35], [-0.8], [0.9], [-0.35]])  # output weights choose slope changes.
c3_w = -0.25                                       # output bias shifts the curve vertically.
y3_w = H3_w @ a3_w + c3_w                          # weighted sum of nonlinear pieces.

print("prediction range:", round(float(y3_w.min()), 3), "to", round(float(y3_w.max()), 3))

plt.figure(figsize=(5, 3))
plt.plot(x3_w[:, 0], y3_w[:, 0], color="teal", label="sum of ReLU pieces")
for j3_w in range(H3_w.shape[1]):
    plt.plot(x3_w[:, 0], 0.15 * H3_w[:, j3_w] - 1.1, alpha=0.55)
plt.title("3: basis pieces add into a flexible curve"); plt.xlabel("x"); plt.legend(); plt.show()

▶ What you'll see: several shifted ramps underneath and their combined curve above them.

*Why it's done this way:* addition is what turns local basis pieces into a global function. The hidden layer supplies a vocabulary of bends; the output weights decide whether each bend should raise, lower, steepen, or flatten the final curve.

### 4. Width controls approximation error

The theorem says that for enough hidden units, the approximation can become arbitrarily accurate. We can see the mechanism with linear interpolation: approximate a continuous curve by many small line segments. A ReLU network can represent those piecewise-linear shapes, so increasing the number of breakpoints reduces the maximum error.

In [ ]:
x4_w = np.linspace(-1, 1, 401)
y4_w = np.sin(3 * x4_w) + 0.3 * x4_w ** 2
widths4_w = np.array([4, 8, 16, 32])
errors4_w = []
for m4_w in widths4_w:
    knots4_w = np.linspace(-1, 1, m4_w)
    vals4_w = np.sin(3 * knots4_w) + 0.3 * knots4_w ** 2
    approx4_w = np.interp(x4_w, knots4_w, vals4_w)
    errors4_w.append(np.max(np.abs(y4_w - approx4_w)))

print("max errors:", np.round(errors4_w, 4))

assert errors4_w[-1] < errors4_w[0]

▶ What you'll see: the maximum absolute error shrinks as the number of knots increases.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(widths4_w, errors4_w, marker="o", color="purple")
plt.title("4: more pieces reduce uniform error")
plt.xlabel("number of knots / pieces"); plt.ylabel("max |target - approx|"); plt.show()

▶ What you'll see: a downward error curve; width buys a finer cover of the target function.

*Why it's done this way:* continuous functions on compact intervals are uniformly continuous, meaning small enough input intervals force small output changes. More hidden units create more intervals, so the largest local mismatch can be pushed down.

### 5. Existence is not the same as easy training

Universal approximation is an **existence theorem**. It says suitable parameters exist; it does not say gradient descent will find them quickly, stably, or with little data. We demonstrate that by training only the output weights for a fixed random ReLU feature bank: the basis is expressive enough to improve, but the final fit depends on the chosen features and conditioning.

In [ ]:
x5_w = np.linspace(-1, 1, 201)[:, None]
y5_w = np.sin(3 * x5_w) + 0.3 * x5_w ** 2
rng5_w = np.random.default_rng(5)
W5_w = rng5_w.normal(0, 3.0, size=(1, 24))
b5_w = rng5_w.uniform(-2.0, 2.0, size=(1, 24))
H5_w = np.maximum(0.0, x5_w @ W5_w + b5_w)

print("feature matrix:", H5_w.shape, "rank:", np.linalg.matrix_rank(H5_w))

▶ What you'll see: a 201×24 nonlinear feature matrix; it is wide but not automatically perfect.

In [ ]:
X5_w = np.c_[H5_w, np.ones(len(x5_w))]        # add output bias as a final column.
theta5_w = np.linalg.solve(X5_w.T @ X5_w + 1e-4 * np.eye(X5_w.shape[1]), X5_w.T @ y5_w)
pred5_w = X5_w @ theta5_w
rmse5_w = float(np.sqrt(np.mean((pred5_w - y5_w) ** 2)))

print("fixed-feature RMSE:", round(rmse5_w, 4))

assert rmse5_w < 0.08
plt.figure(figsize=(5, 3))
plt.plot(x5_w[:, 0], y5_w[:, 0], color="black", label="target")
plt.plot(x5_w[:, 0], pred5_w[:, 0], color="crimson", label="trained output layer")
plt.title("5: existence versus the trained fit"); plt.xlabel("x"); plt.legend(); plt.show()

▶ What you'll see: a close fit, but not a theorem-proof guarantee that every optimizer path succeeds.

*Why it's done this way:* solving the output layer isolates one part of the problem: even with nonlinear features, training quality depends on feature placement, regularization, and numerical stability. The theorem promises representational capacity, not optimization success.

### 6. Capacity must still be constrained

A very wide network can approximate many functions, including noise. That flexibility is useful only when paired with constraints: enough data, regularization, validation, and stable scale. A tiny noisy example shows the danger: a highly flexible interpolating curve can pass through noisy observations while wandering away from the true underlying function.

In [ ]:
rng6_w = np.random.default_rng(6)
x6_w = np.linspace(-1, 1, 18)
true6_w = np.sin(3 * x6_w)
y6_w = true6_w + 0.15 * rng6_w.normal(size=len(x6_w))
coef6_w = np.polyfit(x6_w, y6_w, deg=12)  # deliberately high-capacity polynomial surrogate.
grid6_w = np.linspace(-1, 1, 300)
pred6_w = np.polyval(coef6_w, grid6_w)

print("training RMSE:", round(float(np.sqrt(np.mean((np.polyval(coef6_w, x6_w) - y6_w) ** 2))), 4))

▶ What you'll see: the training error is tiny because the model is flexible enough to chase noise.

In [ ]:
true_grid6_w = np.sin(3 * grid6_w)
grid_rmse6_w = float(np.sqrt(np.mean((pred6_w - true_grid6_w) ** 2)))

print("true-curve RMSE:", round(grid_rmse6_w, 4))

assert grid_rmse6_w > 0.05
plt.figure(figsize=(5, 3))
plt.scatter(x6_w, y6_w, color="black", label="noisy data")
plt.plot(grid6_w, true_grid6_w, color="seagreen", label="true function")
plt.plot(grid6_w, pred6_w, color="red", label="too flexible fit")
plt.ylim(-1.6, 1.6); plt.title("6: approximation capacity can overfit")
plt.xlabel("x"); plt.legend(); plt.show()

▶ What you'll see: the flexible curve follows the noisy points but can oscillate away from the true curve.

*Why it's done this way:* universal approximation removes one worry — expressiveness — but leaves generalization and optimization. Wide networks need constraints because the same capacity that fits the signal can also fit accidental noise.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each toy uses
> small NumPy arrays, prints every intermediate with a real `# ->` value, draws one picture, and
> ends with an `assert` that pins the result.

### ✍️ Toy 1 · Compact continuous targets have small local changes

Universal approximation starts with a continuous target on a bounded interval. On a tiny grid, nearby
inputs create nearby outputs.

In [ ]:
import numpy as np                              # arrays and numerical functions.
import matplotlib.pyplot as plt                 # one picture per toy.

t1_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t1_x = np.linspace(-1.0, 1.0, 9)                # -> [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]

print("x grid:", np.round(t1_x, 2).tolist())    # -> [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]

t1_y = np.sin(np.pi * t1_x / 2) + 0.25 * t1_x   # -> [-1.25, -1.111, -0.832, -0.445, 0.0, 0.445, 0.832, 1.111, 1.25]

print("target values:", np.round(t1_y, 3).tolist())  # -> [-1.25, -1.111, -0.832, -0.445, 0.0, 0.445, 0.832, 1.111, 1.25]

t1_jumps = np.abs(np.diff(t1_y))                # -> [0.139, 0.279, 0.387, 0.445, 0.445, 0.387, 0.279, 0.139]

print("neighbor jumps:", np.round(t1_jumps, 3).tolist())  # -> [0.139, 0.279, 0.387, 0.445, 0.445, 0.387, 0.279, 0.139]

t1_max_jump = t1_jumps.max()                    # -> 0.445

print("max neighbor jump:", round(float(t1_max_jump), 3))  # -> 0.445

assert round(float(t1_max_jump), 3) == 0.445

plt.figure(figsize=(4.8, 2.8))
plt.plot(t1_x, t1_y, marker="o", color="black")
plt.title("Toy 1 · continuous target on [-1, 1]")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.show()

▶ What you'll see: a smooth bounded curve whose neighboring grid values change gradually.

### ✍️ Toy 2 · A ReLU basis piece starts at its breakpoint

One ReLU hidden unit is zero before its affine signal crosses zero, then becomes a ramp.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t2_x = np.linspace(-1.0, 1.0, 9)

print("x grid:", np.round(t2_x, 2).tolist())    # -> [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]

t2_w = 2.0
t2_b = -0.5
t2_z = t2_w * t2_x + t2_b                      # -> [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]

print("affine z:", np.round(t2_z, 2).tolist())  # -> [-2.5, -2.0, -1.5, -1.0, -0.5, 0.0, 0.5, 1.0, 1.5]

t2_h = np.maximum(0.0, t2_z)                    # -> [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.0, 1.5]

print("ReLU basis:", np.round(t2_h, 2).tolist())  # -> [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 1.0, 1.5]

t2_break = -t2_b / t2_w                         # -> 0.25

print("breakpoint:", round(float(t2_break), 3)) # -> 0.25

assert round(float(t2_break), 3) == 0.25

plt.figure(figsize=(4.8, 2.8))
plt.plot(t2_x, t2_z, "--", label="affine z")
plt.plot(t2_x, t2_h, marker="o", label="ReLU(z)")
plt.axvline(t2_break, color="gray", linestyle=":", label="breakpoint")
plt.title("Toy 2 · one adjustable basis ramp")
plt.xlabel("x")
plt.legend()
plt.show()

▶ What you'll see: the hidden unit stays flat until `x = 0.25`, then rises linearly.

### ✍️ Toy 3 · Weighted ReLU pieces add into bends

A hidden layer supplies several ramps, and output weights add or subtract them into one piecewise curve.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t3_x = np.linspace(-1.0, 1.0, 9)

print("x grid:", np.round(t3_x, 2).tolist())    # -> [-1.0, -0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0]

t3_centers = np.array([-0.5, 0.0, 0.5])
t3_w = np.ones(3) * 3.0
t3_b = -t3_w * t3_centers
t3_Z = t3_x[:, None] * t3_w[None, :] + t3_b[None, :]

print("z at x=0:", np.round(t3_Z[4], 3).tolist())       # -> [1.5, 0.0, -1.5]

t3_H = np.maximum(0.0, t3_Z)

print("basis at x=0:", np.round(t3_H[4], 3).tolist())   # -> [1.5, 0.0, 0.0]

t3_a = np.array([0.5, -1.0, 0.75])
t3_parts = t3_H * t3_a

print("weighted parts at x=0:", np.round(t3_parts[4], 3).tolist())  # -> [0.75, -0.0, 0.0]

t3_y = t3_parts.sum(axis=1)                    # -> [0.0, 0.0, 0.0, 0.375, 0.75, 0.375, 0.0, 0.188, 0.375]

print("combined curve:", np.round(t3_y, 3).tolist())  # -> [0.0, 0.0, 0.0, 0.375, 0.75, 0.375, 0.0, 0.188, 0.375]

assert round(float(t3_y[4]), 3) == 0.75

plt.figure(figsize=(4.8, 2.8))
plt.plot(t3_x, t3_y, marker="o", color="teal", label="sum")
plt.plot(t3_x, t3_H * 0.15 - 0.25, alpha=0.6)
plt.title("Toy 3 · ramps add into bends")
plt.xlabel("x")
plt.legend()
plt.show()

▶ What you'll see: three shifted ramps combine into a curve with visible bends.

### ✍️ Toy 4 · More pieces reduce interpolation error

A wider piecewise-linear approximation can place more knots, which lowers the largest mismatch on a
fixed tiny grid.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t4_x = np.linspace(-1.0, 1.0, 11)
t4_y = np.sin(np.pi * t4_x / 2)                 # -> [-1.0, -0.951, -0.809, -0.588, -0.309, 0.0, 0.309, 0.588, 0.809, 0.951, 1.0]

print("target:", np.round(t4_y, 3).tolist())    # -> [-1.0, -0.951, -0.809, -0.588, -0.309, 0.0, 0.309, 0.588, 0.809, 0.951, 1.0]

t4_widths = np.array([3, 5, 9])
t4_errors = []
for t4_m in t4_widths:
    t4_knots = np.linspace(-1.0, 1.0, t4_m)
    t4_vals = np.sin(np.pi * t4_knots / 2)
    t4_pred = np.interp(t4_x, t4_knots, t4_vals)
    t4_err = np.max(np.abs(t4_y - t4_pred))
    t4_errors.append(t4_err)

print("widths:", t4_widths.tolist())            # -> [3, 5, 9]
print("max errors:", np.round(t4_errors, 4).tolist())  # -> [0.209, 0.0682, 0.0152]

assert t4_errors[-1] < t4_errors[0]

plt.figure(figsize=(4.8, 2.8))
plt.plot(t4_widths, t4_errors, marker="o", color="purple")
plt.title("Toy 4 · width lowers max error")
plt.xlabel("number of knots")
plt.ylabel("max error")
plt.show()

▶ What you'll see: the max error drops from about `0.209` to `0.0152` as knots increase.

### ✍️ Toy 5 · Fixed nonlinear features need a solved output layer

Even with expressive random ReLU features, the final fit comes from solving output weights and depends
on the feature matrix.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t5_x = np.linspace(-1.0, 1.0, 9)[:, None]
t5_y = np.sin(np.pi * t5_x / 2)

print("target:", np.round(t5_y[:, 0], 3).tolist())       # -> [-1.0, -0.924, -0.707, -0.383, 0.0, 0.383, 0.707, 0.924, 1.0]

t5_W = t5_rng.normal(0.0, 2.0, size=(1, 6))

print("hidden weights:", np.round(t5_W[0], 3).tolist())  # -> [0.251, -0.264, 1.281, 0.21, -1.071, 0.723]

t5_b = t5_rng.uniform(-1.0, 1.0, size=(1, 6))

print("hidden biases:", np.round(t5_b[0], 3).tolist())   # -> [0.213, 0.459, 0.087, 0.87, 0.632, -0.995]

t5_H = np.maximum(0.0, t5_x @ t5_W + t5_b)

print("feature shape:", t5_H.shape)             # -> (9, 6)

t5_X = np.c_[t5_H, np.ones(t5_H.shape[0])]

print("design shape:", t5_X.shape)              # -> (9, 7)

t5_reg = 1e-3 * np.eye(t5_X.shape[1])
t5_theta = np.linalg.solve(t5_X.T @ t5_X + t5_reg, t5_X.T @ t5_y)

print("output weights:", np.round(t5_theta[:, 0], 3).tolist())  # -> [1.909, 1.415, 0.361, -0.82, -1.062, 0.0, 0.246]

t5_pred = t5_X @ t5_theta
t5_rmse = np.sqrt(np.mean((t5_pred - t5_y) ** 2))

print("RMSE:", round(float(t5_rmse), 4))         # -> 0.0499

assert float(t5_rmse) < 0.06

plt.figure(figsize=(4.8, 2.8))
plt.plot(t5_x[:, 0], t5_y[:, 0], marker="o", color="black", label="target")
plt.plot(t5_x[:, 0], t5_pred[:, 0], marker="s", color="crimson", label="fixed-feature fit")
plt.title("Toy 5 · trained output layer")
plt.xlabel("x")
plt.legend()
plt.show()

▶ What you'll see: fixed random ReLU features can fit the tiny target fairly well after solving the output layer.

### ✍️ Toy 6 · Excess capacity can chase noise

A flexible polynomial surrogate can drive training error near zero while drifting from the underlying
smooth curve between training points.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)               # seeded generator for reproducibility.
t6_x = np.linspace(-1.0, 1.0, 9)
t6_true = np.sin(np.pi * t6_x)                  # -> [-0.0, -0.707, -1.0, -0.707, 0.0, 0.707, 1.0, 0.707, 0.0]

print("true values:", np.round(t6_true, 3).tolist())      # -> [-0.0, -0.707, -1.0, -0.707, 0.0, 0.707, 1.0, 0.707, 0.0]

t6_noise = 0.12 * t6_rng.normal(size=t6_x.size)            # -> [0.015, -0.016, 0.077, 0.013, -0.064, 0.043, 0.156, 0.114, -0.084]

print("noise:", np.round(t6_noise, 3).tolist())            # -> [0.015, -0.016, 0.077, 0.013, -0.064, 0.043, 0.156, 0.114, -0.084]

t6_y = t6_true + t6_noise

print("noisy data:", np.round(t6_y, 3).tolist())           # -> [0.015, -0.723, -0.923, -0.695, -0.064, 0.75, 1.156, 0.821, -0.084]

t6_coef = np.polyfit(t6_x, t6_y, deg=6)

print("poly coefficients:", np.round(t6_coef, 3).tolist()) # -> [1.72, 1.295, -3.055, -4.478, 1.355, 3.134, -0.054]

t6_train_pred = np.polyval(t6_coef, t6_x)
t6_train_rmse = np.sqrt(np.mean((t6_train_pred - t6_y) ** 2))

print("training RMSE:", round(float(t6_train_rmse), 4))    # -> 0.0079

t6_holdout_x = np.linspace(-0.875, 0.875, 8)
t6_holdout_true = np.sin(np.pi * t6_holdout_x)
t6_holdout_pred = np.polyval(t6_coef, t6_holdout_x)
t6_holdout_rmse = np.sqrt(np.mean((t6_holdout_pred - t6_holdout_true) ** 2))

print("true-curve RMSE:", round(float(t6_holdout_rmse), 4))  # -> 0.082

assert float(t6_holdout_rmse) > float(t6_train_rmse)

plt.figure(figsize=(4.8, 2.8))
plt.scatter(t6_x, t6_y, color="black", label="noisy data")
plt.plot(t6_x, t6_true, color="seagreen", label="true")
plt.plot(t6_x, t6_train_pred, color="red", label="flexible fit")
plt.title("Toy 6 · capacity can fit noise")
plt.xlabel("x")
plt.legend()
plt.show()

▶ What you'll see: the flexible fit nearly hits the noisy training points, but its true-curve error is larger.


## 🛠️ Setup

In [ ]:
import numpy as np  # Import NumPy for arrays, linear algebra, activations, and small neural-network computations.
import matplotlib.pyplot as plt  # Import Matplotlib for the plots that make approximation behavior visible.
np.random.seed(0)  # Fix the global random seed so every stochastic example is repeatable.

## 🟢 Basics (warm-up)

### Basic 1 — Plot the compact-domain target

**Goal.** Create a continuous target on a bounded interval, because universal approximation is a statement about functions on compact domains. We build it in 2 steps.

In [ ]:
x_b1 = np.linspace(-1.0, 1.0, 101)[:, None]  # Create a compact input grid as a column vector.
y_b1 = np.sin(3 * x_b1) + 0.3 * x_b1 ** 2  # Define a smooth target that has both wave and curvature.

print("x range:", float(x_b1.min()), "to", float(x_b1.max()))  # Inspect the closed input interval.
print("y shape:", y_b1.shape)  # Inspect that there is one target value per input.

▶ What you'll see: a one-dimensional supervised dataset sampled on `[-1, 1]`.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact target-function figure.
plt.plot(x_b1[:, 0], y_b1[:, 0], color="black")  # Plot the continuous target samples.
plt.title("Basic 1: target function")  # Label the warm-up plot.
plt.xlabel("x")  # Label the input axis.
plt.ylabel("f(x)")  # Label the output axis.
plt.show()  # Display the curve.

▶ What you'll see: a smooth curve that a network will try to imitate.

👀 Takeaway: the theorem needs a bounded input region and a continuous target, not an arbitrary function everywhere.

### Basic 2 — Compute one affine signal

**Goal.** Compute `z = wx + b` for one neuron, because every hidden unit starts by measuring a signed distance from a movable threshold. We build it in 2 steps.

In [ ]:
x_b2 = np.array([[-0.5], [0.0], [0.5]])  # Pick three inputs so the arithmetic is inspectable.
w_b2 = 2.0  # Choose a scalar weight that controls slope.
b_b2 = -0.25  # Choose a scalar bias that shifts the zero crossing.
z_b2 = w_b2 * x_b2 + b_b2  # Compute the affine pre-activation.

print("z values:", z_b2[:, 0])  # Inspect raw neuron signals before nonlinearity.

▶ What you'll see: the affine signal is negative for small x and positive for larger x.

In [ ]:
break_b2 = -b_b2 / w_b2  # Solve wx+b=0 to locate the threshold.

print("breakpoint:", round(break_b2, 3))  # Inspect where the affine signal changes sign.

assert round(break_b2, 3) == 0.125  # Verify the threshold calculation.
plt.figure(figsize=(4, 3))  # Create a compact affine plot.
plt.plot(x_b2[:, 0], z_b2[:, 0], marker="o", color="teal")  # Visualize z across the three inputs.
plt.axhline(0, color="black", linewidth=0.8)  # Show the activation threshold.
plt.title("Basic 2: affine signal")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.ylabel("z")  # Label pre-activation.
plt.show()  # Display the line.

▶ What you'll see: the line crosses zero near `x = 0.125`.

👀 Takeaway: weights and biases position each neuron's breakpoint before the nonlinearity acts.

### Basic 3 — Apply a ReLU nonlinearity

**Goal.** Turn affine values into a nonlinear feature, because a network without nonlinearities collapses to one global linear map. We build it in 2 steps.

In [ ]:
z_b3 = np.array([-2.0, -0.5, 0.0, 0.5, 2.0])  # Define pre-activations around the threshold.
h_b3 = np.maximum(0.0, z_b3)  # Apply ReLU by clipping negative values to zero.

print("z:", z_b3)  # Inspect the input to the activation.
print("ReLU(z):", h_b3)  # Inspect the activated output.

▶ What you'll see: all negative entries become zero while positive entries are unchanged.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a small activation plot.
plt.plot(z_b3, h_b3, marker="o", color="purple")  # Plot ReLU as a piecewise-linear function.
plt.title("Basic 3: ReLU nonlinearity")  # Title the activation plot.
plt.xlabel("z")  # Label pre-activation.
plt.ylabel("max(0,z)")  # Label activation.
plt.show()  # Display the curve.

▶ What you'll see: a flat zero region followed by a line with slope 1.

👀 Takeaway: nonlinearity is the source of bendable function shapes.

### Basic 4 — Build one hidden unit over many inputs

**Goal.** Evaluate a single ReLU unit on a full input grid, because hidden units are basis functions over the domain. We build it in 2 steps.

In [ ]:
x_b4 = np.linspace(-1, 1, 101)[:, None]  # Create a compact input grid.
w_b4 = 5.0  # Use a positive slope for the affine signal.
b_b4 = 1.0  # Shift the unit so its breakpoint is left of zero.
h_b4 = np.maximum(0.0, w_b4 * x_b4 + b_b4)  # Evaluate the hidden unit on all inputs.

print("hidden min/max:", float(h_b4.min()), round(float(h_b4.max()), 3))  # Inspect activation range.

▶ What you'll see: the unit is zero on part of the domain and positive elsewhere.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a compact hidden-unit plot.
plt.plot(x_b4[:, 0], h_b4[:, 0], color="darkorange")  # Draw the ramp-like basis function.
plt.title("Basic 4: one ReLU basis function")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.ylabel("hidden activation")  # Label activation.
plt.show()  # Display the basis piece.

▶ What you'll see: one ramp that begins at a learned threshold.

👀 Takeaway: a hidden unit is a movable nonlinear basis piece.

### Basic 5 — Stack several hidden units

**Goal.** Compute a hidden activation matrix `H`, because a width-m network evaluates m basis pieces at every input. We build it in 2 steps.

In [ ]:
x_b5 = np.linspace(-1, 1, 51)[:, None]  # Create input samples.
W_b5 = np.array([[3.0, 3.0, 3.0]])  # Use three hidden weights as columns.
b_b5 = np.array([[1.5, 0.0, -1.5]])  # Place three different breakpoints.
H_b5 = np.maximum(0.0, x_b5 @ W_b5 + b_b5)  # Compute all hidden activations at once.

print("H shape:", H_b5.shape)  # Inspect samples by hidden units.

assert H_b5.shape == (51, 3)  # Verify the hidden matrix dimensions.

▶ What you'll see: the hidden layer has one column per basis function.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a basis-family plot.
for j_b5 in range(H_b5.shape[1]):  # Draw each hidden unit separately.
    plt.plot(x_b5[:, 0], H_b5[:, j_b5], label=f"unit {j_b5}")  # Plot one column of H.
plt.title("Basic 5: width means multiple basis pieces")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.legend()  # Show unit labels.
plt.show()  # Display the basis functions.

▶ What you'll see: three ramps starting at three different locations.

👀 Takeaway: width gives the network a richer dictionary of nonlinear pieces.

### Basic 6 — Add basis pieces with output weights

**Goal.** Combine hidden activations with output weights, because the network output is a weighted sum of learned features. We build it in 2 steps.

In [ ]:
x_b6 = np.linspace(-1, 1, 101)[:, None]  # Create inputs for the network output.
W_b6 = np.array([[4.0, 4.0, 4.0]])  # Define hidden slopes.
b_b6 = np.array([[2.0, 0.0, -2.0]])  # Define shifted breakpoints.
a_b6 = np.array([[0.4], [-0.8], [0.4]])  # Define output weights that add and subtract pieces.
H_b6 = np.maximum(0.0, x_b6 @ W_b6 + b_b6)  # Compute hidden features.
yhat_b6 = H_b6 @ a_b6  # Add basis pieces into one output.

print("output shape:", yhat_b6.shape)  # Inspect that one prediction is produced per input.

▶ What you'll see: a single output column created from three hidden columns.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a network-output plot.
plt.plot(x_b6[:, 0], yhat_b6[:, 0], color="teal")  # Plot the weighted sum.
plt.title("Basic 6: weighted sum of hidden pieces")  # Title the output plot.
plt.xlabel("x")  # Label input.
plt.ylabel("network output")  # Label prediction.
plt.show()  # Display the curve.

▶ What you'll see: adding ramps creates a curve with several slope changes.

👀 Takeaway: the output layer turns hidden basis activations into the approximating function.

### Basic 7 — Measure approximation error

**Goal.** Compute mean squared error and max error, because approximation quality needs a numeric distance from the target. We build it in 2 steps.

In [ ]:
x_b7 = np.linspace(-1, 1, 101)[:, None]  # Create input samples.
y_b7 = np.sin(3 * x_b7)  # Define a target curve.
rough_b7 = 0.9 * x_b7  # Define a crude linear approximation.
err_b7 = y_b7 - rough_b7  # Compute pointwise residuals.

print("first residuals:", np.round(err_b7[:5, 0], 3))  # Inspect local errors.

▶ What you'll see: residuals vary across the input domain.

In [ ]:
mse_b7 = float(np.mean(err_b7 ** 2))  # Average squared residuals.
maxerr_b7 = float(np.max(np.abs(err_b7)))  # Compute worst-case absolute residual.

print("MSE:", round(mse_b7, 4), "max error:", round(maxerr_b7, 4))  # Inspect two error summaries.

assert maxerr_b7 > 0.0  # Verify the approximation is not exact.
plt.figure(figsize=(4, 3))  # Create an error plot.
plt.plot(x_b7[:, 0], err_b7[:, 0], color="crimson")  # Draw residuals over x.
plt.axhline(0, color="black", linewidth=0.8)  # Mark zero error.
plt.title("Basic 7: residual curve")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.ylabel("target - approximation")  # Label residual.
plt.show()  # Display the residual curve.

▶ What you'll see: the line misses the target by different amounts at different x-values.

👀 Takeaway: universal approximation is about making these residuals uniformly small with enough capacity.

### Basic 8 — Use least squares on fixed nonlinear features

**Goal.** Fit output weights for fixed hidden units, because once nonlinear features are chosen, the last layer is a linear regression problem. We build it in 3 steps.

In [ ]:
x_b8 = np.linspace(-1, 1, 121)[:, None]  # Create training inputs.
y_b8 = np.sin(3 * x_b8) + 0.3 * x_b8 ** 2  # Create target outputs.
centers_b8 = np.linspace(-1, 1, 9)  # Place nine feature breakpoints across the domain.
H_b8 = np.maximum(0.0, 6.0 * (x_b8 - centers_b8.reshape(1, -1)))  # Build fixed ReLU features.

print("feature matrix:", H_b8.shape)  # Inspect samples by features.

▶ What you'll see: a 121×9 feature matrix for the fixed hidden layer.

In [ ]:
X_b8 = np.c_[H_b8, np.ones(len(x_b8))]  # Add a bias column for vertical shift.
theta_b8 = np.linalg.lstsq(X_b8, y_b8, rcond=None)[0]  # Solve least squares for output weights.
pred_b8 = X_b8 @ theta_b8  # Compute fitted predictions.
rmse_b8 = float(np.sqrt(np.mean((pred_b8 - y_b8) ** 2)))  # Measure fit quality.

print("RMSE:", round(rmse_b8, 4))  # Inspect the output-layer fit.

assert rmse_b8 < 0.15  # Verify the nonlinear feature bank fits reasonably well.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a fit plot.
plt.plot(x_b8[:, 0], y_b8[:, 0], color="black", label="target")  # Draw target.
plt.plot(x_b8[:, 0], pred_b8[:, 0], color="teal", label="fixed-feature fit")  # Draw fit.
plt.title("Basic 8: fit output weights")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.legend()  # Show curve labels.
plt.show()  # Display the comparison.

▶ What you'll see: the weighted ReLU features track the target curve much better than a single line.

👀 Takeaway: nonlinear hidden features make the final linear combination powerful.

### Basic 9 — Compare linear and nonlinear approximation

**Goal.** Compare a line with a small ReLU feature model, because nonlinearity is what lets a network bend. We build it in 3 steps.

In [ ]:
x_b9 = np.linspace(-1, 1, 121)[:, None]  # Create inputs.
y_b9 = np.sin(3 * x_b9)  # Define a nonlinear target.
Xlin_b9 = np.c_[x_b9, np.ones(len(x_b9))]  # Build linear features x and bias.
theta_lin_b9 = np.linalg.lstsq(Xlin_b9, y_b9, rcond=None)[0]  # Fit the best line.
pred_lin_b9 = Xlin_b9 @ theta_lin_b9  # Compute linear predictions.

print("linear theta:", np.round(theta_lin_b9[:, 0], 3))  # Inspect slope and intercept.

▶ What you'll see: the best line has only two parameters.

In [ ]:
centers_b9 = np.linspace(-1, 1, 11)  # Choose more breakpoints for the nonlinear model.
H_b9 = np.maximum(0.0, 7.0 * (x_b9 - centers_b9.reshape(1, -1)))  # Compute ReLU features.
Xrelu_b9 = np.c_[H_b9, np.ones(len(x_b9))]  # Add bias to nonlinear features.
theta_relu_b9 = np.linalg.lstsq(Xrelu_b9, y_b9, rcond=None)[0]  # Fit output weights.
pred_relu_b9 = Xrelu_b9 @ theta_relu_b9  # Compute nonlinear predictions.
rmse_lin_b9 = float(np.sqrt(np.mean((pred_lin_b9 - y_b9) ** 2)))  # Linear error.
rmse_relu_b9 = float(np.sqrt(np.mean((pred_relu_b9 - y_b9) ** 2)))  # ReLU-feature error.

print("linear RMSE:", round(rmse_lin_b9, 4), "ReLU RMSE:", round(rmse_relu_b9, 4))  # Compare errors.

assert rmse_relu_b9 < rmse_lin_b9  # Verify nonlinearity improves the fit.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a comparison plot.
plt.plot(x_b9[:, 0], y_b9[:, 0], color="black", label="target")  # Draw target.
plt.plot(x_b9[:, 0], pred_lin_b9[:, 0], color="gray", label="line")  # Draw linear fit.
plt.plot(x_b9[:, 0], pred_relu_b9[:, 0], color="purple", label="ReLU features")  # Draw nonlinear fit.
plt.title("Basic 9: line vs nonlinear features")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.legend()  # Show labels.
plt.show()  # Display the comparison.

▶ What you'll see: the ReLU-feature curve bends with the target while the line cannot.

👀 Takeaway: stacking affine maps without nonlinear activation would not create universal approximation.

### Basic 10 — Watch width reduce error

**Goal.** Sweep the number of fixed ReLU features, because more hidden units give more places to bend the approximation. We build it in 3 steps.

In [ ]:
x_b10 = np.linspace(-1, 1, 161)[:, None]  # Create a common training grid.
y_b10 = np.sin(3 * x_b10) + 0.3 * x_b10 ** 2  # Define the target curve.
widths_b10 = np.array([3, 5, 9, 17])  # Choose hidden widths to compare.
rmse_b10 = []  # Prepare storage for fit errors.

print("widths:", widths_b10)  # Inspect the capacity sweep.

▶ What you'll see: the sweep will compare four hidden-layer sizes.

In [ ]:
for m_b10 in widths_b10:  # Fit one fixed-feature model per width.
    centers_b10 = np.linspace(-1, 1, m_b10)  # Spread breakpoints across the compact interval.
    H_b10 = np.maximum(0.0, 6.0 * (x_b10 - centers_b10.reshape(1, -1)))  # Compute hidden features.
    X_b10 = np.c_[H_b10, np.ones(len(x_b10))]  # Add output bias.
    theta_b10 = np.linalg.lstsq(X_b10, y_b10, rcond=None)[0]  # Fit output weights.
    pred_b10 = X_b10 @ theta_b10  # Compute predictions.
    rmse_b10.append(float(np.sqrt(np.mean((pred_b10 - y_b10) ** 2))))  # Store RMSE.

print("RMSE by width:", np.round(rmse_b10, 4))  # Inspect approximation improvement.

assert rmse_b10[-1] < rmse_b10[0]  # Verify wider feature bank fits better here.

In [ ]:
plt.figure(figsize=(4, 3))  # Create a width-vs-error plot.
plt.plot(widths_b10, rmse_b10, marker="o", color="seagreen")  # Plot error against hidden width.
plt.title("Basic 10: width lowers approximation error")  # Title the sweep.
plt.xlabel("hidden width")  # Label capacity axis.
plt.ylabel("RMSE")  # Label error axis.
plt.show()  # Display the curve.

▶ What you'll see: error decreases as the hidden layer gets more basis pieces.

👀 Takeaway: width is the approximation-capacity knob in the theorem.

## 🟡 Easy

### Easy 1 — Approximate a smooth target with fixed ReLU features

**Goal.** Build a one-hidden-layer ReLU approximation end-to-end, because the theorem's formula is a weighted sum of nonlinear basis functions. We build it in 4 steps.

In [ ]:
x_e1 = np.linspace(-1, 1, 201)[:, None]  # Create compact-domain inputs.
y_e1 = np.sin(3 * x_e1) + 0.3 * x_e1 ** 2  # Define a continuous target.
centers_e1 = np.linspace(-1, 1, 21)  # Place many breakpoints across the domain.

print("training samples:", len(x_e1), "hidden units:", len(centers_e1))  # Inspect data and width.

▶ What you'll see: a small supervised regression problem with 21 hidden basis pieces.

In [ ]:
H_e1 = np.maximum(0.0, 8.0 * (x_e1 - centers_e1.reshape(1, -1)))  # Compute ReLU hidden activations.
X_e1 = np.c_[H_e1, np.ones(len(x_e1))]  # Add a bias feature for the output layer.
theta_e1 = np.linalg.solve(X_e1.T @ X_e1 + 1e-5 * np.eye(X_e1.shape[1]), X_e1.T @ y_e1)  # Fit regularized output weights.

print("theta shape:", theta_e1.shape)  # Inspect one output weight per feature plus bias.

In [ ]:
pred_e1 = X_e1 @ theta_e1  # Compute network predictions.
rmse_e1 = float(np.sqrt(np.mean((pred_e1 - y_e1) ** 2)))  # Measure average error.
maxerr_e1 = float(np.max(np.abs(pred_e1 - y_e1)))  # Measure worst visible error.

print("RMSE:", round(rmse_e1, 4), "max error:", round(maxerr_e1, 4))  # Inspect approximation quality.

assert rmse_e1 < 0.04  # Verify the approximation is close.

In [ ]:
plt.figure(figsize=(5, 3))  # Create the approximation plot.
plt.plot(x_e1[:, 0], y_e1[:, 0], color="black", label="target")  # Draw target.
plt.plot(x_e1[:, 0], pred_e1[:, 0], color="teal", label="ReLU network")  # Draw fitted network.
plt.title("Easy 1: one-hidden-layer approximation")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.legend()  # Show labels.
plt.show()  # Display the fit.

▶ What you'll see: the ReLU network closely tracks the smooth target over the full interval.

👀 Takeaway: a weighted sum of nonlinear hidden units can approximate a smooth function on a compact domain.

### Easy 2 — Approximate a step-like but continuous ramp

**Goal.** Fit a steep continuous transition, because universal approximation covers continuous functions even when they change quickly. We build it in 4 steps.

In [ ]:
x_e2 = np.linspace(-1, 1, 241)[:, None]  # Create inputs.
y_e2 = 1.0 / (1.0 + np.exp(-12.0 * x_e2))  # Define a steep but continuous sigmoid target.

print("target endpoints:", round(float(y_e2[0]), 3), round(float(y_e2[-1]), 3))  # Inspect range.

▶ What you'll see: the target moves from near 0 to near 1 across the interval.

In [ ]:
centers_e2 = np.linspace(-1, 1, 31)  # Use many breakpoints near the steep transition.
H_e2 = np.maximum(0.0, 10.0 * (x_e2 - centers_e2.reshape(1, -1)))  # Fixed ReLU feature bank.
X_e2 = np.c_[H_e2, np.ones(len(x_e2))]  # Add output bias.
theta_e2 = np.linalg.solve(X_e2.T @ X_e2 + 1e-4 * np.eye(X_e2.shape[1]), X_e2.T @ y_e2)  # Fit output weights.

print("feature columns:", X_e2.shape[1])  # Inspect model width including bias.

In [ ]:
pred_e2 = X_e2 @ theta_e2  # Predict the sigmoid-like target.
rmse_e2 = float(np.sqrt(np.mean((pred_e2 - y_e2) ** 2)))  # Compute error.

print("RMSE:", round(rmse_e2, 4))  # Inspect fit quality.

assert rmse_e2 < 0.02  # Verify the fit is accurate on the sampled grid.

In [ ]:
plt.figure(figsize=(5, 3))  # Create the steep-transition plot.
plt.plot(x_e2[:, 0], y_e2[:, 0], color="black", label="target")  # Draw target.
plt.plot(x_e2[:, 0], pred_e2[:, 0], color="darkorange", label="ReLU approximation")  # Draw approximation.
plt.title("Easy 2: approximating a steep continuous change")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.legend()  # Show labels.
plt.show()  # Display the fit.

▶ What you'll see: the approximation bends sharply near zero without needing an actual discontinuity.

👀 Takeaway: fast-changing continuous functions need enough local pieces where the change happens.

### Easy 3 — Show why a single linear layer is not enough

**Goal.** Fit the same target with a linear model and a nonlinear feature model, because nonlinear activation is what prevents the network from collapsing to a line. We build it in 4 steps.

In [ ]:
x_e3 = np.linspace(-1, 1, 181)[:, None]  # Create inputs.
y_e3 = np.cos(4 * x_e3)  # Define a curved target that a line cannot follow.
Xlin_e3 = np.c_[x_e3, np.ones(len(x_e3))]  # Build linear features.
theta_lin_e3 = np.linalg.lstsq(Xlin_e3, y_e3, rcond=None)[0]  # Fit the best linear model.
pred_lin_e3 = Xlin_e3 @ theta_lin_e3  # Compute linear predictions.

print("linear weights:", np.round(theta_lin_e3[:, 0], 3))  # Inspect line parameters.

▶ What you'll see: the linear model has only slope and intercept.

In [ ]:
centers_e3 = np.linspace(-1, 1, 25)  # Define nonlinear breakpoints.
H_e3 = np.maximum(0.0, 9.0 * (x_e3 - centers_e3.reshape(1, -1)))  # Compute hidden features.
Xrelu_e3 = np.c_[H_e3, np.ones(len(x_e3))]  # Add output bias.
theta_relu_e3 = np.linalg.solve(Xrelu_e3.T @ Xrelu_e3 + 1e-4 * np.eye(Xrelu_e3.shape[1]), Xrelu_e3.T @ y_e3)  # Fit nonlinear output.
pred_relu_e3 = Xrelu_e3 @ theta_relu_e3  # Compute nonlinear predictions.

print("nonlinear feature columns:", Xrelu_e3.shape[1])  # Inspect capacity.

In [ ]:
rmse_lin_e3 = float(np.sqrt(np.mean((pred_lin_e3 - y_e3) ** 2)))  # Linear RMSE.
rmse_relu_e3 = float(np.sqrt(np.mean((pred_relu_e3 - y_e3) ** 2)))  # Nonlinear RMSE.

print("linear RMSE:", round(rmse_lin_e3, 4), "ReLU RMSE:", round(rmse_relu_e3, 4))  # Compare errors.

assert rmse_relu_e3 < 0.5 * rmse_lin_e3  # Verify nonlinear features are much better.

In [ ]:
plt.figure(figsize=(5, 3))  # Create the comparison plot.
plt.plot(x_e3[:, 0], y_e3[:, 0], color="black", label="target")  # Draw target.
plt.plot(x_e3[:, 0], pred_lin_e3[:, 0], color="gray", label="linear")  # Draw linear fit.
plt.plot(x_e3[:, 0], pred_relu_e3[:, 0], color="purple", label="nonlinear")  # Draw nonlinear fit.
plt.title("Easy 3: nonlinearity creates bends")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.legend()  # Show labels.
plt.show()  # Display the comparison.

▶ What you'll see: the nonlinear model follows multiple bends, while the linear model averages them away.

👀 Takeaway: affine layers alone compose into another affine layer; activation functions are essential.

### Easy 4 — Train output weights with gradient descent

**Goal.** Optimize output weights by gradient descent instead of a closed-form solve, because practical neural networks learn parameters iteratively. We build it in 4 steps.

In [ ]:
x_e4 = np.linspace(-1, 1, 151)[:, None]  # Create training inputs.
y_e4 = np.sin(2.5 * x_e4)  # Define target outputs.
centers_e4 = np.linspace(-1, 1, 15)  # Define fixed hidden-unit breakpoints.
H_e4 = np.maximum(0.0, 7.0 * (x_e4 - centers_e4.reshape(1, -1)))  # Compute fixed features.
X_e4 = np.c_[H_e4, np.ones(len(x_e4))]  # Add bias feature.
theta_e4 = np.zeros((X_e4.shape[1], 1))  # Initialize output weights at zero.

print("parameters:", theta_e4.size)  # Inspect learned output parameters.

▶ What you'll see: only the output layer will be optimized in this example.

In [ ]:
lr_e4 = 0.001  # Choose a stable learning rate for squared loss.
losses_e4 = []  # Store loss over time.
for step_e4 in range(600):  # Repeat small gradient steps.
    pred_e4 = X_e4 @ theta_e4  # Compute current predictions.
    err_e4 = pred_e4 - y_e4  # Compute residuals as prediction minus target.
    grad_e4 = (2.0 / len(x_e4)) * X_e4.T @ err_e4  # Gradient of mean squared error.
    theta_e4 -= lr_e4 * grad_e4  # Move weights opposite the gradient.
    if step_e4 % 20 == 0:  # Record a readable learning curve.
        losses_e4.append(float(np.mean(err_e4 ** 2)))  # Save MSE.

print("loss start -> end:", round(losses_e4[0], 4), "->", round(losses_e4[-1], 4))  # Inspect progress.

assert losses_e4[-1] < losses_e4[0]  # Verify training reduced loss.

In [ ]:
pred_final_e4 = X_e4 @ theta_e4  # Compute final predictions.
rmse_e4 = float(np.sqrt(np.mean((pred_final_e4 - y_e4) ** 2)))  # Compute final RMSE.

print("final RMSE:", round(rmse_e4, 4))  # Inspect fit quality.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a training-curve plot.
plt.plot(np.arange(len(losses_e4)) * 20, losses_e4, color="teal")  # Plot MSE over steps.
plt.title("Easy 4: gradient descent lowers output loss")  # Title the plot.
plt.xlabel("step")  # Label optimization step.
plt.ylabel("MSE")  # Label loss.
plt.show()  # Display the curve.

▶ What you'll see: the loss decreases steadily as output weights learn how to combine features.

👀 Takeaway: approximation capacity must be paired with an optimizer that actually finds useful weights.

### Easy 5 — Visualize underfitting and overfitting

**Goal.** Compare low, medium, and high capacity fits on noisy samples, because universal approximation does not remove the need for validation. We build it in 4 steps.

In [ ]:
rng_e5 = np.random.default_rng(55)  # Create reproducible noisy data.
x_e5 = np.linspace(-1, 1, 24)[:, None]  # Use a small training set.
true_e5 = np.sin(3 * x_e5)  # Define the noiseless signal.
y_e5 = true_e5 + 0.12 * rng_e5.normal(size=true_e5.shape)  # Add observation noise.

print("training points:", len(x_e5))  # Inspect sample size.

▶ What you'll see: a small dataset where excessive flexibility can chase noise.

In [ ]:
grid_e5 = np.linspace(-1, 1, 240)[:, None]  # Create a dense grid for visualization.
widths_e5 = [3, 9, 25]  # Compare low, medium, and high feature counts.
preds_e5 = []  # Store grid predictions.
train_rmse_e5 = []  # Store training errors.
for m_e5 in widths_e5:  # Fit each capacity level.
    centers_e5 = np.linspace(-1, 1, m_e5)  # Place breakpoints.
    Htrain_e5 = np.maximum(0.0, 8.0 * (x_e5 - centers_e5.reshape(1, -1)))  # Training features.
    Xtrain_e5 = np.c_[Htrain_e5, np.ones(len(x_e5))]  # Add bias.
    theta_e5 = np.linalg.solve(Xtrain_e5.T @ Xtrain_e5 + 1e-5 * np.eye(Xtrain_e5.shape[1]), Xtrain_e5.T @ y_e5)  # Fit output.
    Hgrid_e5 = np.maximum(0.0, 8.0 * (grid_e5 - centers_e5.reshape(1, -1)))  # Grid features.
    preds_e5.append(np.c_[Hgrid_e5, np.ones(len(grid_e5))] @ theta_e5)  # Store grid fit.
    train_rmse_e5.append(float(np.sqrt(np.mean((Xtrain_e5 @ theta_e5 - y_e5) ** 2))))  # Store training error.

print("train RMSE:", np.round(train_rmse_e5, 4))  # Inspect fit to noisy samples.

In [ ]:
assert train_rmse_e5[-1] <= train_rmse_e5[0]  # Verify larger capacity can fit training points at least as well.

print("lowest training RMSE width:", widths_e5[int(np.argmin(train_rmse_e5))])  # Inspect which capacity fits noisy data best.

In [ ]:
plt.figure(figsize=(5, 3))  # Create capacity comparison plot.
plt.scatter(x_e5[:, 0], y_e5[:, 0], color="black", label="noisy data")  # Plot training observations.
plt.plot(grid_e5[:, 0], np.sin(3 * grid_e5[:, 0]), color="seagreen", label="true signal")  # Draw true signal.
for pred_e5, m_e5 in zip(preds_e5, widths_e5):  # Draw each capacity fit.
    plt.plot(grid_e5[:, 0], pred_e5[:, 0], label=f"width {m_e5}")  # Plot model curve.
plt.title("Easy 5: capacity changes the fit")  # Title the comparison.
plt.xlabel("x")  # Label input.
plt.ylim(-1.5, 1.5)  # Keep oscillations visible.
plt.legend()  # Show labels.
plt.show()  # Display the plot.

▶ What you'll see: low width underfits, higher width follows the data more closely, and too much flexibility may chase noise.

👀 Takeaway: universal approximation is about representational power, not automatic generalization.

## 🔴 Advanced

### Advanced 1 — Approximate a two-input continuous surface

**Goal.** Extend the idea from curves to surfaces, because universal approximation applies to multivariate continuous functions on compact sets. We build it in 5 steps.

In [ ]:
side_a1 = 31  # Choose a small square grid.
u_a1 = np.linspace(-1, 1, side_a1)  # First input coordinate.
v_a1 = np.linspace(-1, 1, side_a1)  # Second input coordinate.
U_a1, V_a1 = np.meshgrid(u_a1, v_a1)  # Create a compact 2-D domain.
X_a1 = np.c_[U_a1.ravel(), V_a1.ravel()]  # Flatten grid into input rows.
y_a1 = (np.sin(2 * X_a1[:, [0]]) + np.cos(3 * X_a1[:, [1]])) / 2.0  # Continuous surface target.

print("X shape:", X_a1.shape, "y shape:", y_a1.shape)  # Inspect multivariate data.

▶ What you'll see: 961 two-dimensional input points and one target value per point.

In [ ]:
rng_a1 = np.random.default_rng(101)  # Reproducible random feature bank.
W_a1 = rng_a1.normal(size=(2, 80))  # Random directions for 80 hidden units.
b_a1 = rng_a1.uniform(-1.5, 1.5, size=(1, 80))  # Random offsets for hidden units.
H_a1 = np.maximum(0.0, X_a1 @ W_a1 + b_a1)  # Compute ReLU features over the 2-D domain.

print("hidden matrix:", H_a1.shape)  # Inspect samples by hidden units.

In [ ]:
Xfeat_a1 = np.c_[H_a1, np.ones(len(X_a1))]  # Add output bias.
theta_a1 = np.linalg.solve(Xfeat_a1.T @ Xfeat_a1 + 1e-3 * np.eye(Xfeat_a1.shape[1]), Xfeat_a1.T @ y_a1)  # Fit output weights.
pred_a1 = Xfeat_a1 @ theta_a1  # Predict the surface values.
rmse_a1 = float(np.sqrt(np.mean((pred_a1 - y_a1) ** 2)))  # Measure approximation error.

print("surface RMSE:", round(rmse_a1, 4))  # Inspect fit quality.

assert rmse_a1 < 0.08  # Verify the random-feature approximation is useful.

In [ ]:
Ztrue_a1 = y_a1.reshape(side_a1, side_a1)  # Reshape target to an image grid.
Zpred_a1 = pred_a1.reshape(side_a1, side_a1)  # Reshape prediction to an image grid.

print("center true/pred:", round(float(Ztrue_a1[side_a1//2, side_a1//2]), 3), round(float(Zpred_a1[side_a1//2, side_a1//2]), 3))  # Inspect one location.

In [ ]:
fig_a1, ax_a1 = plt.subplots(1, 2, figsize=(7, 3))  # Create side-by-side surface images.
ax_a1[0].imshow(Ztrue_a1, extent=[-1, 1, -1, 1], origin="lower", cmap="viridis")  # Show target surface.
ax_a1[0].set_title("target surface")  # Title target.
ax_a1[1].imshow(Zpred_a1, extent=[-1, 1, -1, 1], origin="lower", cmap="viridis")  # Show approximation.
ax_a1[1].set_title("ReLU approximation")  # Title prediction.
plt.suptitle("Advanced 1: two-input universal approximation")  # Overall title.
plt.show()  # Display images.

▶ What you'll see: the predicted heatmap resembles the smooth two-dimensional target surface.

👀 Takeaway: hidden units form nonlinear regions in input space, and weighted sums approximate multivariate surfaces too.

### Advanced 2 — Compare activation choices

**Goal.** Fit the same target with ReLU and tanh feature banks, because the theorem needs a non-polynomial nonlinear activation but the shape affects efficiency. We build it in 5 steps.

In [ ]:
x_a2 = np.linspace(-1, 1, 201)[:, None]  # Create compact-domain inputs.
y_a2 = np.sin(4 * x_a2) + 0.2 * np.cos(9 * x_a2)  # Define a wavy continuous target.
centers_a2 = np.linspace(-1, 1, 25)  # Shared feature centers.

print("target std:", round(float(np.std(y_a2)), 3))  # Inspect target scale.

▶ What you'll see: a continuous target with both slow and fast variation.

In [ ]:
H_relu_a2 = np.maximum(0.0, 9.0 * (x_a2 - centers_a2.reshape(1, -1)))  # ReLU feature bank.
H_tanh_a2 = np.tanh(5.0 * (x_a2 - centers_a2.reshape(1, -1)))  # Tanh feature bank.

print("feature shapes:", H_relu_a2.shape, H_tanh_a2.shape)  # Inspect equal-width banks.

In [ ]:
X_relu_a2 = np.c_[H_relu_a2, np.ones(len(x_a2))]  # Add bias to ReLU features.
X_tanh_a2 = np.c_[H_tanh_a2, np.ones(len(x_a2))]  # Add bias to tanh features.
theta_relu_a2 = np.linalg.solve(X_relu_a2.T @ X_relu_a2 + 1e-4 * np.eye(X_relu_a2.shape[1]), X_relu_a2.T @ y_a2)  # Fit ReLU output.
theta_tanh_a2 = np.linalg.solve(X_tanh_a2.T @ X_tanh_a2 + 1e-4 * np.eye(X_tanh_a2.shape[1]), X_tanh_a2.T @ y_a2)  # Fit tanh output.
pred_relu_a2 = X_relu_a2 @ theta_relu_a2  # ReLU predictions.
pred_tanh_a2 = X_tanh_a2 @ theta_tanh_a2  # Tanh predictions.

print("fits computed")  # Confirm both models were fit.

In [ ]:
rmse_relu_a2 = float(np.sqrt(np.mean((pred_relu_a2 - y_a2) ** 2)))  # ReLU RMSE.
rmse_tanh_a2 = float(np.sqrt(np.mean((pred_tanh_a2 - y_a2) ** 2)))  # Tanh RMSE.

print("ReLU RMSE:", round(rmse_relu_a2, 4), "tanh RMSE:", round(rmse_tanh_a2, 4))  # Compare activations.

assert min(rmse_relu_a2, rmse_tanh_a2) < 0.08  # Verify at least one nonlinear bank approximates well.

In [ ]:
plt.figure(figsize=(5, 3))  # Create activation comparison plot.
plt.plot(x_a2[:, 0], y_a2[:, 0], color="black", label="target")  # Draw target.
plt.plot(x_a2[:, 0], pred_relu_a2[:, 0], color="teal", label="ReLU")  # Draw ReLU fit.
plt.plot(x_a2[:, 0], pred_tanh_a2[:, 0], color="orange", label="tanh")  # Draw tanh fit.
plt.title("Advanced 2: activation shape affects efficiency")  # Title the plot.
plt.xlabel("x")  # Label input.
plt.legend()  # Show labels.
plt.show()  # Display the comparison.

▶ What you'll see: both nonlinear activations can bend, but their approximation errors may differ at the same width.

👀 Takeaway: universal approximation is broad, yet activation choice still changes numerical efficiency and trainability.

### Advanced 3 — Train both hidden and output weights by backprop

**Goal.** Optimize a small one-hidden-layer network from scratch, because real networks usually learn the basis pieces instead of fixing them. We build it in 5 steps.

In [ ]:
x_a3 = np.linspace(-1, 1, 121)[:, None]  # Create training inputs.
y_a3 = np.sin(3 * x_a3)  # Define target outputs.
rng_a3 = np.random.default_rng(303)  # Reproducible initialization.
W1_a3 = rng_a3.normal(0, 0.8, size=(1, 16))  # Hidden weights.
b1_a3 = np.zeros((1, 16))  # Hidden biases.
W2_a3 = rng_a3.normal(0, 0.2, size=(16, 1))  # Output weights.
b2_a3 = np.zeros((1, 1))  # Output bias.

print("parameters:", W1_a3.size + b1_a3.size + W2_a3.size + b2_a3.size)  # Inspect trainable count.

▶ What you'll see: a tiny neural network with learned hidden and output parameters.

In [ ]:
losses_a3 = []  # Store MSE over training.
lr_a3 = 0.03  # Choose a modest learning rate.
for step_a3 in range(1800):  # Run full-batch gradient descent.
    Z1_a3 = x_a3 @ W1_a3 + b1_a3  # Hidden pre-activations.
    H_a3 = np.maximum(0.0, Z1_a3)  # ReLU hidden activations.
    pred_a3 = H_a3 @ W2_a3 + b2_a3  # Network predictions.
    err_a3 = pred_a3 - y_a3  # Prediction residuals.
    if step_a3 % 60 == 0:  # Record a readable learning curve.
        losses_a3.append(float(np.mean(err_a3 ** 2)))  # Save MSE.
    d_pred_a3 = (2.0 / len(x_a3)) * err_a3  # Derivative of mean squared error.
    dW2_a3 = H_a3.T @ d_pred_a3  # Gradient for output weights.
    db2_a3 = np.sum(d_pred_a3, axis=0, keepdims=True)  # Gradient for output bias.
    dH_a3 = d_pred_a3 @ W2_a3.T  # Backpropagate into hidden activations.
    dZ1_a3 = dH_a3 * (Z1_a3 > 0)  # ReLU derivative gates gradients.
    dW1_a3 = x_a3.T @ dZ1_a3  # Gradient for hidden weights.
    db1_a3 = np.sum(dZ1_a3, axis=0, keepdims=True)  # Gradient for hidden biases.
    W2_a3 -= lr_a3 * dW2_a3; b2_a3 -= lr_a3 * db2_a3  # Update output layer.
    W1_a3 -= lr_a3 * dW1_a3; b1_a3 -= lr_a3 * db1_a3  # Update hidden layer.

print("loss start -> end:", round(losses_a3[0], 4), "->", round(losses_a3[-1], 4))  # Inspect progress.

assert losses_a3[-1] < losses_a3[0]  # Verify training improved the network.

In [ ]:
Z1_final_a3 = x_a3 @ W1_a3 + b1_a3  # Final hidden pre-activations.
H_final_a3 = np.maximum(0.0, Z1_final_a3)  # Final hidden activations.
pred_final_a3 = H_final_a3 @ W2_a3 + b2_a3  # Final predictions.
rmse_a3 = float(np.sqrt(np.mean((pred_final_a3 - y_a3) ** 2)))  # Final RMSE.

print("final RMSE:", round(rmse_a3, 4))  # Inspect fit quality.

assert rmse_a3 < 0.35  # Verify the trained model learned a useful approximation.

In [ ]:
plt.figure(figsize=(5, 3))  # Create a learning-curve plot.
plt.plot(np.arange(len(losses_a3)) * 60, losses_a3, color="purple")  # Plot MSE over training.
plt.title("Advanced 3: backprop lowers approximation loss")  # Title the plot.
plt.xlabel("step")  # Label optimization step.
plt.ylabel("MSE")  # Label loss.
plt.show()  # Display the curve.

▶ What you'll see: the loss decreases as both breakpoints and output weights adapt.

In [ ]:
plt.figure(figsize=(5, 3))  # Create final fit plot.
plt.plot(x_a3[:, 0], y_a3[:, 0], color="black", label="target")  # Draw target.
plt.plot(x_a3[:, 0], pred_final_a3[:, 0], color="crimson", label="trained network")  # Draw network prediction.
plt.title("Advanced 3: learned hidden basis")  # Title the fit plot.
plt.xlabel("x")  # Label input.
plt.legend()  # Show labels.
plt.show()  # Display final approximation.

▶ What you'll see: the trained network captures the main shape, but the optimizer does not prove optimality.

👀 Takeaway: backprop searches for an approximating set of parameters; the theorem only says some good set exists.

### Advanced 4 — Study conditioning of wide feature banks

**Goal.** Compare feature-bank conditioning as width grows, because wide networks can be expressive and numerically awkward at the same time. We build it in 4 steps.

In [ ]:
x_a4 = np.linspace(-1, 1, 151)[:, None]  # Create inputs.
y_a4 = np.sin(3 * x_a4) + 0.2 * x_a4  # Define a smooth target.
widths_a4 = np.array([5, 11, 21, 41])  # Choose increasing widths.
rmse_a4 = []  # Store approximation error.
conds_a4 = []  # Store condition numbers.

print("widths:", widths_a4)  # Inspect the sweep.

▶ What you'll see: the experiment will test more and more hidden features.

In [ ]:
for m_a4 in widths_a4:  # Fit a model at each width.
    centers_a4 = np.linspace(-1, 1, m_a4)  # Spread breakpoints evenly.
    H_a4 = np.maximum(0.0, 8.0 * (x_a4 - centers_a4.reshape(1, -1)))  # ReLU feature bank.
    X_a4 = np.c_[H_a4, np.ones(len(x_a4))]  # Add output bias.
    theta_a4 = np.linalg.solve(X_a4.T @ X_a4 + 1e-6 * np.eye(X_a4.shape[1]), X_a4.T @ y_a4)  # Fit output weights with tiny ridge.
    pred_a4 = X_a4 @ theta_a4  # Compute predictions.
    rmse_a4.append(float(np.sqrt(np.mean((pred_a4 - y_a4) ** 2))))  # Store error.
    conds_a4.append(float(np.linalg.cond(X_a4.T @ X_a4 + 1e-6 * np.eye(X_a4.shape[1]))))  # Store conditioning.

print("RMSE:", np.round(rmse_a4, 5))  # Inspect fit quality.
print("condition numbers:", np.round(conds_a4, 1))  # Inspect numerical scale.

In [ ]:
assert rmse_a4[-1] <= rmse_a4[0]  # Verify extra width does not hurt the regularized fit here.
assert conds_a4[-1] > conds_a4[0]  # Verify wider correlated features can worsen conditioning.

print("conditioning ratio:", round(conds_a4[-1] / conds_a4[0], 1))  # Inspect growth in numerical difficulty.

In [ ]:
fig_a4, ax_a4 = plt.subplots(1, 2, figsize=(7, 3))  # Create side-by-side diagnostics.
ax_a4[0].plot(widths_a4, rmse_a4, marker="o", color="teal")  # Plot error by width.
ax_a4[0].set_title("fit error")  # Title first panel.
ax_a4[0].set_xlabel("width")  # Label width.
ax_a4[0].set_ylabel("RMSE")  # Label error.
ax_a4[1].plot(widths_a4, conds_a4, marker="o", color="crimson")  # Plot condition number by width.
ax_a4[1].set_title("conditioning")  # Title second panel.
ax_a4[1].set_xlabel("width")  # Label width.
ax_a4[1].set_yscale("log")  # Log scale for large condition numbers.
plt.suptitle("Advanced 4: expressiveness vs numerical scale")  # Overall title.
plt.show()  # Display diagnostics.

▶ What you'll see: approximation error can fall while the linear solve becomes more ill-conditioned.

👀 Takeaway: wide networks need numerical care; expressiveness alone is not stable training.

### Advanced 5 — Validate capacity instead of trusting training fit

**Goal.** Choose width using held-out data, because approximation power must be balanced against noisy generalization. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(505)  # Create reproducible noisy regression data.
x_all_a5 = np.linspace(-1, 1, 80)[:, None]  # Create all inputs.
y_true_a5 = np.sin(3 * x_all_a5)  # Define true signal.
y_all_a5 = y_true_a5 + 0.12 * rng_a5.normal(size=y_true_a5.shape)  # Add noise.
train_mask_a5 = np.arange(len(x_all_a5)) % 4 != 0  # Hold out every fourth point.
x_train_a5, y_train_a5 = x_all_a5[train_mask_a5], y_all_a5[train_mask_a5]  # Training split.
x_val_a5, y_val_a5 = x_all_a5[~train_mask_a5], y_all_a5[~train_mask_a5]  # Validation split.

print("train/val sizes:", len(x_train_a5), len(x_val_a5))  # Inspect split sizes.

▶ What you'll see: the noisy data is split so capacity can be checked on unseen points.

In [ ]:
widths_a5 = np.array([3, 7, 15, 31, 55])  # Candidate hidden widths.
train_rmse_a5 = []  # Store training errors.
val_rmse_a5 = []  # Store validation errors.

print("candidate widths:", widths_a5)  # Inspect capacity choices.

In [ ]:
for m_a5 in widths_a5:  # Fit one model per width.
    centers_a5 = np.linspace(-1, 1, m_a5)  # Place feature centers.
    Htr_a5 = np.maximum(0.0, 9.0 * (x_train_a5 - centers_a5.reshape(1, -1)))  # Training features.
    Xtr_a5 = np.c_[Htr_a5, np.ones(len(x_train_a5))]  # Add bias to training features.
    theta_a5 = np.linalg.solve(Xtr_a5.T @ Xtr_a5 + 1e-4 * np.eye(Xtr_a5.shape[1]), Xtr_a5.T @ y_train_a5)  # Fit output weights.
    Hval_a5 = np.maximum(0.0, 9.0 * (x_val_a5 - centers_a5.reshape(1, -1)))  # Validation features.
    Xval_a5 = np.c_[Hval_a5, np.ones(len(x_val_a5))]  # Add bias to validation features.
    train_rmse_a5.append(float(np.sqrt(np.mean((Xtr_a5 @ theta_a5 - y_train_a5) ** 2))))  # Store train RMSE.
    val_rmse_a5.append(float(np.sqrt(np.mean((Xval_a5 @ theta_a5 - y_val_a5) ** 2))))  # Store validation RMSE.

print("train RMSE:", np.round(train_rmse_a5, 4))  # Inspect training curve.
print("val RMSE:", np.round(val_rmse_a5, 4))  # Inspect validation curve.

In [ ]:
best_idx_a5 = int(np.argmin(val_rmse_a5))  # Find the width with lowest validation error.
best_width_a5 = int(widths_a5[best_idx_a5])  # Read selected width.

print("best validation width:", best_width_a5)  # Inspect model selection result.

assert val_rmse_a5[best_idx_a5] <= max(val_rmse_a5)  # Verify the selected width is not worse than all alternatives.

In [ ]:
plt.figure(figsize=(5, 3))  # Create model-selection plot.
plt.plot(widths_a5, train_rmse_a5, marker="o", label="train RMSE")  # Plot training error.
plt.plot(widths_a5, val_rmse_a5, marker="o", label="validation RMSE")  # Plot validation error.
plt.axvline(best_width_a5, color="red", linestyle="--", label=f"best width={best_width_a5}")  # Mark selected width.
plt.title("Advanced 5: validate the capacity knob")  # Title the plot.
plt.xlabel("hidden width")  # Label capacity axis.
plt.ylabel("RMSE")  # Label error.
plt.legend()  # Show labels.
plt.show()  # Display the selection curve.

▶ What you'll see: training error generally rewards larger width, while validation chooses the width that generalizes best.

👀 Takeaway: universal approximation tells us wide networks can fit; validation tells us how wide they should be for noisy data.